# exp018 — Cabeza por hallazgo sobre la entrada v2 (12 queries, 6 slots)

SPEC: `SPEC.md` en esta carpeta. Padre: exp012_entrada_v2. Misma receta y entrada; solo cambia la cabeza (una query por hallazgo, como exp013). Bitácora en `BITACORA.md`.

## 0. Parámetros (única celda con valores)

In [ ]:
EXP = "exp018_cabeza_hallazgo_v2"
PADRE = "exp012"
FOLDS = [0]
SEED = 42
SPLIT = "split_v1_seed42"
SELECT_ON = "pseudo"
SMOKE = True

LABELS_CSV = "/workspace/data/labels/report_labels_v1.csv"
CACHE = "v2_05mm"
CACHE_ROOT = f"/workspace/data/cache/{CACHE}"
STUDY_META_CSV = "/workspace/data/cache/meta_v2/study_meta_v2.csv"
BACKBONE_PATH = "/workspace/models/dinov2-small"

IMG_SIZE = 336
N_SLOTS = 6
N_SLICES = 16
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 8
FREEZE_EPOCHS = 1
LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0
N_WORKERS = 8
NOTES = "cabeza por hallazgo (12 queries) sobre entrada v2"

## 1. Setup

In [ ]:
import json
import math
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(REPO))
from src.knee.config import LABELS, current_user
from src.knee.data import KneeDataset
from src.knee.experiments import decision, git_info, guardar_metricas, leer_metricas, seed_everything
from src.knee.metrics import mean_auc
from src.knee.model import KneeModel
from src.knee.splits import load_split

assert SELECT_ON == "pseudo", "Gold nunca selecciona checkpoints (docs/06_SDD.md)"
assert (REPO / "experiments" / EXP / "SPEC.md").exists(), "Primero escribe la SPEC del experimento"
AUTHOR = current_user()
RESULTS = REPO / "results" / EXP
RESULTS.mkdir(parents=True, exist_ok=True)
GIT = git_info(REPO)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if SMOKE:
    EPOCHS, FREEZE_EPOCHS = 1, 0
seed_everything(SEED)
print(f"{EXP} · {AUTHOR} · git {GIT['commit']} ({GIT['branch']}) · {torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu'}")
if GIT["src_dirty"]:
    print("OJO: src/ o SPLITS/ tienen cambios sin commit; el resultado no será reproducible.")

## 2. Datos y split (las revisiones detienen el notebook si algo está mal)

In [ ]:
import cv2
from torch.utils.data import Dataset

SLOT_COLUMNS = [
    "slot1_sag_fluid_fs",
    "slot2_cor_fluid_fs",
    "slot3_ax_fluid_fs",
    "slot4_sag_pdt2",
    "slot5_cor_t1",
    "slot6_sag_t1",
]

def sample_slice_indices(total, count, train=False):
    if total <= 0:
        return np.zeros(count, dtype=int)
    positions = np.linspace(0, total - 1, count)
    if train and total > count:
        step = (total - 1) / max(count - 1, 1)
        positions += np.random.uniform(-0.35 * step, 0.35 * step, size=count)
    return np.clip(np.round(positions), 0, total - 1).astype(int)

def prepare_slices(volume, n_slices, image_size, train=False):
    indices = sample_slice_indices(len(volume), n_slices, train)
    images = np.stack([
        cv2.resize(volume[index], (image_size, image_size), interpolation=cv2.INTER_AREA)
        for index in indices
    ]).astype(np.float32)
    if train:
        gain = np.random.uniform(0.9, 1.1)
        offset = np.random.uniform(-8, 8)
        angle = np.random.uniform(-5, 5)
        scale = np.random.uniform(0.97, 1.03)
        matrix = cv2.getRotationMatrix2D((image_size / 2, image_size / 2), angle, scale)
        images = np.stack([
            cv2.warpAffine(image, matrix, (image_size, image_size), borderMode=cv2.BORDER_REFLECT_101)
            for image in images
        ])
        images = np.clip(images * gain + offset, 0, 255)
    images = images[:, None] / 255.0
    images = np.repeat(images, 3, axis=1)
    mean = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)[None, :, None, None]
    std = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)[None, :, None, None]
    return torch.from_numpy((images - mean) / std).float()

class KneeV2Dataset(Dataset):
    def __init__(self, studies, cache_root, study_meta, labels=None, n_slots=6, n_slices=16,
                 image_size=336, train=False):
        self.studies = [str(study) for study in studies]
        self.cache_root = Path(cache_root)
        meta = pd.read_csv(study_meta, dtype={"study": str}) if not isinstance(study_meta, pd.DataFrame) else study_meta.copy()
        meta["study"] = meta.study.astype(str)
        missing_columns = set(SLOT_COLUMNS[:n_slots]) - set(meta.columns)
        if missing_columns:
            raise ValueError(f"Faltan columnas de slots: {sorted(missing_columns)}")
        self.study_meta = meta.set_index("study")
        self.n_slots = n_slots
        self.n_slices = n_slices
        self.image_size = image_size
        self.train = train
        if labels is None:
            self.targets = None
        else:
            labels_df = pd.read_csv(labels) if not isinstance(labels, pd.DataFrame) else labels.copy()
            labels_df["StudyInstanceUID"] = labels_df.StudyInstanceUID.astype(str)
            self.targets = labels_df.set_index("StudyInstanceUID")

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        study = self.studies[index]
        if study not in self.study_meta.index:
            raise KeyError(f"El estudio {study} no existe en study_meta_v2.csv")
        row = self.study_meta.loc[study]
        images = torch.zeros(self.n_slots, self.n_slices, 3, self.image_size, self.image_size)
        series_meta = torch.arange(self.n_slots, dtype=torch.long)
        series_mask = torch.ones(self.n_slots, dtype=torch.bool)
        for slot_index, column in enumerate(SLOT_COLUMNS[:self.n_slots]):
            series = row[column]
            if pd.isna(series) or not str(series).strip():
                continue
            path = self.cache_root / study / f"{series}.npy"
            volume = np.load(path, mmap_mode="r")
            if volume.ndim != 3 or volume.dtype != np.uint8:
                raise ValueError(f"Cache v2 invalido en {path}: shape={volume.shape}, dtype={volume.dtype}")
            if row.get("study_side") == "L":
                volume = volume[::-1] if slot_index in (0, 3, 5) else volume[:, :, ::-1]
            images[slot_index] = prepare_slices(volume, self.n_slices, self.image_size, self.train)
            series_mask[slot_index] = False
        if series_mask.all():
            raise RuntimeError(f"El estudio {study} no tiene slots disponibles")

        target = torch.full((len(LABELS),), float("nan"))
        if self.targets is not None and study in self.targets.index:
            values = pd.to_numeric(self.targets.loc[study, LABELS], errors="coerce").to_numpy(dtype=np.float32)
            target = torch.from_numpy(values)
        return {"study": study, "images": images, "series_meta": series_meta,
                "series_mask": series_mask, "targets": target}

split = load_split(SPLIT)
pseudo = pd.read_csv(LABELS_CSV)
pseudo["StudyInstanceUID"] = pseudo.StudyInstanceUID.astype(str)
pseudo = pseudo[["StudyInstanceUID", *LABELS]]
train_csv = pd.read_csv("/workspace/data/raw/train.csv", encoding="utf-8")
gold = train_csv.dropna(subset=LABELS, how="all")[["StudyInstanceUID", *LABELS]]
gold["StudyInstanceUID"] = gold.StudyInstanceUID.astype(str)

pool = split[split.fold >= 0].merge(pseudo, on="StudyInstanceUID", how="inner")
assert len(gold) == 58 and set(gold.StudyInstanceUID) == set(split.loc[split.is_gold == 1, "StudyInstanceUID"])
assert not set(gold.StudyInstanceUID) & set(pool.StudyInstanceUID), "gold dentro del entrenamiento"
assert set(FOLDS) <= set(range(split.attrs["n_folds"]))

study_meta = pd.read_csv(STUDY_META_CSV, dtype={"study": str})
study_meta["study"] = study_meta.study.astype(str)
train_meta = study_meta[study_meta["split"] == "train"] if "split" in study_meta else study_meta
slot_present = train_meta[SLOT_COLUMNS].notna() & train_meta[SLOT_COLUMNS].astype(str).ne("")
coverage = slot_present.mean()
n_available = slot_present.sum(axis=1)
n_zero = int((n_available == 0).sum())
zero_fraction = n_zero / max(1, len(train_meta))
print("Cobertura por slot:", {column: f"{coverage[column]:.1%}" for column in SLOT_COLUMNS})
print(f"Estudios sin slots: {n_zero}/{len(train_meta)} ({zero_fraction:.2%})")
if zero_fraction >= 0.005:
    raise RuntimeError(f"{zero_fraction:.2%} de estudios no tienen slots; se exige <0.5%")
available_studies = set(train_meta.loc[n_available > 0, "study"])
pool = pool[pool.StudyInstanceUID.isin(available_studies)].reset_index(drop=True)
gold = gold[gold.StudyInstanceUID.isin(available_studies)].reset_index(drop=True)
print(f"pool {len(pool)} estudios · gold {len(gold)} · folds {pool.fold.value_counts().sort_index().to_dict()}")

dataset_args = dict(cache_root=CACHE_ROOT, study_meta=study_meta, n_slots=N_SLOTS, n_slices=N_SLICES, image_size=IMG_SIZE)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=N_WORKERS, pin_memory=True, persistent_workers=N_WORKERS > 0)

def make_loaders(fold):
    train_ids = pool.loc[pool.fold != fold, "StudyInstanceUID"]
    val_ids = pool.loc[pool.fold == fold, "StudyInstanceUID"]
    gold_ids = gold.StudyInstanceUID
    if SMOKE:
        train_ids, val_ids, gold_ids = train_ids[:80], val_ids[:40], gold_ids
    generator = torch.Generator().manual_seed(SEED + fold)
    return (
        DataLoader(KneeV2Dataset(train_ids, labels=pool, train=True, **dataset_args), shuffle=True, generator=generator, **loader_args),
        DataLoader(KneeV2Dataset(val_ids, labels=pool, train=False, **dataset_args), shuffle=False, **loader_args),
        DataLoader(KneeV2Dataset(gold_ids, labels=gold, train=False, **dataset_args), shuffle=False, **loader_args),
    )

## 3. Modelo

In [ ]:
from contextlib import nullcontext
from torch import nn

class AttentionPool(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, values, padding_mask=None):
        query = self.query.expand(values.shape[0], -1, -1)
        pooled, _ = self.attention(query, values, values, key_padding_mask=padding_mask, need_weights=False)
        return pooled[:, 0]

class KneeV2Model(nn.Module):
    def __init__(self, backbone=None, backbone_path=None, feature_dim=None, num_labels=12,
                 attention_heads=4, gradient_checkpointing=False, n_slots=6):
        super().__init__()
        if backbone is None and backbone_path:
            from transformers import Dinov2Model
            backbone = Dinov2Model.from_pretrained(backbone_path, local_files_only=True)
            feature_dim = feature_dim or backbone.config.hidden_size
        if backbone is None or feature_dim is None:
            raise ValueError("Indica backbone + feature_dim o backbone_path")
        self.backbone = backbone
        self.feature_dim = feature_dim
        self.slot_embedding = nn.Embedding(n_slots, feature_dim)
        self.slice_pool = AttentionPool(feature_dim, attention_heads)
        self.series_pool = AttentionPool(feature_dim, attention_heads)
        self.head = nn.Sequential(
            nn.LayerNorm(feature_dim),
            nn.Linear(feature_dim, feature_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(feature_dim, num_labels),
        )
        if gradient_checkpointing and hasattr(self.backbone, "gradient_checkpointing_enable"):
            self.backbone.gradient_checkpointing_enable()

    def freeze_backbone(self, frozen=True):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not frozen

    def autocast_context(self, device):
        if torch.device(device).type == "cuda":
            return torch.autocast("cuda", dtype=torch.bfloat16)
        return nullcontext()

    def _encode(self, images):
        output = self.backbone(pixel_values=images) if hasattr(self.backbone, "config") else self.backbone(images)
        if hasattr(output, "last_hidden_state"):
            return output.last_hidden_state[:, 0]
        if isinstance(output, (tuple, list)):
            output = output[0]
        if output.ndim == 4:
            output = output.mean(dim=(-2, -1))
        return output

    def forward(self, images, series_meta, series_mask):
        batch, n_series, n_slices, channels, height, width = images.shape
        features = self._encode(images.reshape(-1, channels, height, width))
        features = features.reshape(batch * n_series, n_slices, self.feature_dim)
        series_features = self.slice_pool(features).reshape(batch, n_series, self.feature_dim)
        series_features = series_features + self.slot_embedding(series_meta.long())
        study_features = self.series_pool(series_features, padding_mask=series_mask.bool())
        return self.head(study_features)

class LabelQueryPool(nn.Module):
    """Una query aprendida por hallazgo que atiende sobre las series (slots) del estudio (exp013)."""

    def __init__(self, dim, num_labels=12, heads=4):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, num_labels, dim) * 0.02)
        self.attention = nn.MultiheadAttention(dim, heads, batch_first=True)

    def forward(self, values, padding_mask=None):
        query = self.query.expand(values.shape[0], -1, -1)
        pooled, weights = self.attention(query, values, values, key_padding_mask=padding_mask,
                                         need_weights=True, average_attn_weights=True)
        return pooled, weights  # [B, 12, d], [B, 12, S]

class KneeV2LabelQueryModel(KneeV2Model):
    """KneeV2Model de exp012 con la agregación entre series y la cabeza cambiadas (12 queries, una por hallazgo)."""

    def __init__(self, num_labels=12, attention_heads=4, **kwargs):
        super().__init__(num_labels=num_labels, attention_heads=attention_heads, **kwargs)
        dim = self.feature_dim
        del self.series_pool, self.head
        self.label_pool = LabelQueryPool(dim, num_labels, attention_heads)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Dropout(0.2))
        bound = 1 / math.sqrt(dim)  # misma init que un nn.Linear(dim, 1) por etiqueta
        self.label_weight = nn.Parameter(torch.empty(num_labels, dim).uniform_(-bound, bound))
        self.label_bias = nn.Parameter(torch.empty(num_labels).uniform_(-bound, bound))

    def forward(self, images, series_meta, series_mask, return_attention=False):
        batch, n_series, n_slices, channels, height, width = images.shape
        features = self._encode(images.reshape(-1, channels, height, width))
        features = features.reshape(batch * n_series, n_slices, self.feature_dim)
        series_features = self.slice_pool(features).reshape(batch, n_series, self.feature_dim)
        series_features = series_features + self.slot_embedding(series_meta.long())
        # series_mask (v2): True = slot faltante -> se enmascara en la atención
        label_features, attention = self.label_pool(series_features, padding_mask=series_mask.bool())
        hidden = self.head(label_features)
        logits = (hidden * self.label_weight).sum(-1) + self.label_bias
        return (logits, attention) if return_attention else logits

def build_model():
    model = KneeV2LabelQueryModel(backbone_path=BACKBONE_PATH, gradient_checkpointing=True, n_slots=N_SLOTS)
    return model.to(DEVICE)

def build_optimizer(model, steps_per_epoch):
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    optimizer = AdamW([
        {"params": list(model.backbone.parameters()), "lr": BACKBONE_LR},
        {"params": [p for p in model.parameters() if id(p) not in backbone_ids], "lr": LR},
    ], weight_decay=WEIGHT_DECAY)
    total = max(1, steps_per_epoch * EPOCHS)
    warmup = int(total * WARMUP_RATIO)

    def schedule(step):
        if step < warmup:
            return step / max(1, warmup)
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))

    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)

def masked_bce(logits, targets):
    valid = torch.isfinite(targets)
    if not valid.any():
        return logits.sum() * 0.0
    losses = torch.nn.functional.binary_cross_entropy_with_logits(logits, torch.nan_to_num(targets), reduction="none")
    return losses[valid].mean()

## 4. Entrenamiento

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    predictions, targets, studies = [], [], []
    for batch in loader:
        with model.autocast_context(DEVICE):
            logits = model(batch["images"].to(DEVICE), batch["series_meta"].to(DEVICE), batch["series_mask"].to(DEVICE))
        predictions.append(logits.sigmoid().float().cpu().numpy())
        targets.append(batch["targets"].numpy())
        studies.extend(batch["study"])
    predictions, targets = np.concatenate(predictions), np.concatenate(targets)
    frame = pd.DataFrame(predictions, columns=[f"pred_{label}" for label in LABELS])
    frame.insert(0, "StudyInstanceUID", studies)
    return frame, mean_auc(targets, predictions)

def train_fold(fold):
    seed_everything(SEED + fold)
    train_loader, val_loader, gold_loader = make_loaders(fold)
    model = build_model()
    steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    optimizer, scheduler = build_optimizer(model, steps_per_epoch)
    history, best = [], None
    start = time.time()
    for epoch in range(EPOCHS):
        model.freeze_backbone(epoch < FREEZE_EPOCHS)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, batch in enumerate(train_loader):
            with model.autocast_context(DEVICE):
                logits = model(batch["images"].to(DEVICE, non_blocking=True),
                               batch["series_meta"].to(DEVICE, non_blocking=True),
                               batch["series_mask"].to(DEVICE, non_blocking=True))
                loss = masked_bce(logits, batch["targets"].to(DEVICE, non_blocking=True)) / GRAD_ACCUM
            loss.backward()
            running += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        oof, pseudo_metrics = predict(model, val_loader)
        gold_preds, gold_metrics = predict(model, gold_loader)
        history.append({"epoch": epoch + 1, "train_loss": running / len(train_loader),
                        "pseudo_auc": pseudo_metrics["mean"], "gold_auc": gold_metrics["mean"],
                        "minutes": (time.time() - start) / 60})
        pd.DataFrame(history).to_csv(RESULTS / f"history_f{fold}.csv", index=False)
        print(f"fold {fold} epoch {epoch + 1}: loss {history[-1]['train_loss']:.4f} "
              f"pseudo {pseudo_metrics['mean']:.4f} gold {gold_metrics['mean']:.4f} ({history[-1]['minutes']:.1f} min)")
        if best is None or pseudo_metrics["mean"] > best["pseudo"]["mean"]:
            best = {"epoch": epoch + 1, "pseudo": pseudo_metrics, "gold": gold_metrics}
            torch.save({"model": model.state_dict(), "exp": EXP, "fold": fold, "epoch": epoch + 1,
                        "params": {k: v for k, v in PARAMS.items()}}, RESULTS / f"best_f{fold}.pt")
            oof.to_csv(RESULTS / f"oof_f{fold}.csv", index=False)
            gold_preds.to_csv(RESULTS / f"gold_f{fold}.csv", index=False)
    return {"pseudo_auc": best["pseudo"]["mean"], "gold_auc": best["gold"]["mean"], "best_epoch": best["epoch"],
            "pseudo_per_label": best["pseudo"]["per_label"], "gold_per_label": best["gold"]["per_label"],
            "gpu_hours": (time.time() - start) / 3600, "ckpt_path": str(RESULTS / f"best_f{fold}.pt")}

PARAMS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
per_fold = {}
for fold in FOLDS:
    per_fold[str(fold)] = train_fold(fold)

## 5. Evaluación contra el padre

In [ ]:
gold_auc = mean_auc(
    gold.set_index("StudyInstanceUID").loc[:, LABELS].to_numpy(),
    np.mean([pd.read_csv(RESULTS / f"gold_f{f}.csv").set_index("StudyInstanceUID").loc[gold.StudyInstanceUID].to_numpy()
             for f in FOLDS], axis=0),
)
oof_auc = None
if sorted(FOLDS) == list(range(split.attrs["n_folds"])):
    oof = pd.concat([pd.read_csv(RESULTS / f"oof_f{f}.csv") for f in FOLDS]).merge(pool[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID")
    oof_auc = mean_auc(oof[LABELS].to_numpy(), oof[[f"pred_{l}" for l in LABELS]].to_numpy())["mean"]

parent = {}
for parent_root in [REPO / "results", Path(f"/workspace/users/{AUTHOR}/results")]:
    try:  # el metrics.json del padre puede no estar en este clon (exp012 corre en el repo principal del usuario)
        parent = (leer_metricas(PADRE, root=parent_root) if PADRE else None) or {}
    except FileNotFoundError:
        parent = {}
    if parent:
        break
parent_f0 = parent.get("per_fold", {}).get("0", {}).get("pseudo_auc")
ours_f0 = per_fold.get("0", {}).get("pseudo_auc")
delta_pseudo = ours_f0 - parent_f0 if ours_f0 is not None and parent_f0 is not None else None
# screen (solo fold 0): gold contra el gold del fold 0 del padre, no contra su promedio de 5 folds
parent_gold = parent.get("per_fold", {}).get("0", {}).get("gold_auc") if FOLDS == [0] else parent.get("gold_auc")
delta_gold = gold_auc["mean"] - parent_gold if parent_gold is not None else None
DECISION = decision(delta_pseudo, delta_gold)

summary = pd.DataFrame({
    "padre": [parent_f0, parent_gold, parent.get("oof_pseudo_auc")],
    "este": [ours_f0, gold_auc["mean"], oof_auc],
}, index=["pseudo fold 0", "gold (58)", "OOF pseudo 5 folds"])
summary["delta"] = summary.este - summary.padre
display(summary.round(4))
display(pd.DataFrame({f: v["gold_per_label"] for f, v in per_fold.items()}).round(3))
print("Decisión automática:", DECISION)
for f in FOLDS:
    pd.read_csv(RESULTS / f"history_f{f}.csv").plot(x="epoch", y=["pseudo_auc", "gold_auc"], marker="o", title=f"{EXP} fold {f}")
    plt.show()

# --- Atención por hallazgo, agregable por SLOT (instrumentación de exp013; no cambia el modelo) ---
PLANES = ["sagittal", "coronal", "axial"]
SLOT_PLANE = np.array([0, 1, 2, 0, 1, 0])[:N_SLOTS]  # SAG FS, COR FS, AX FS, SAG PD/T2, COR T1, SAG T1
SLOT_NAMES = SLOT_COLUMNS[:N_SLOTS]

@torch.no_grad()
def collect_attention(model, loader):
    model.eval()
    weights, slots, present, studies = [], [], [], []
    for batch in loader:
        with model.autocast_context(DEVICE):
            _, attention = model(batch["images"].to(DEVICE), batch["series_meta"].to(DEVICE),
                                 batch["series_mask"].to(DEVICE), return_attention=True)
        weights.append(attention.float().cpu().numpy())
        slots.append(batch["series_meta"].numpy())
        present.append(~batch["series_mask"].numpy().astype(bool))  # v2: series_mask True = faltante
        studies.extend(batch["study"])
    return np.concatenate(weights), np.concatenate(slots), np.concatenate(present), np.array(studies)

def attention_summary(weights, slots, present):
    # weights [N, 12, S] · slots [N, S] (id de slot de cada token) · present [N, S]
    w = weights * present[:, None, :]
    plane = SLOT_PLANE[slots]
    plane_mass = {name: (w * (plane[:, None, :] == pid)).sum(-1).mean(0) for pid, name in enumerate(PLANES)}
    slot_hit = (slots[:, None, :] == np.arange(N_SLOTS)[None, :, None]).astype(w.dtype)  # [N, slot, S]
    per_slot = np.einsum("nls,nks->nlk", w, slot_hit)  # [N, 12, slot]
    slot_present = (slot_hit * present[:, None, :]).sum(-1) > 0  # [N, slot]
    slot_mass = per_slot.mean(0)  # [12, slot]; slot faltante cuenta como 0
    slot_mass_present = per_slot.sum(0) / np.clip(slot_present.sum(0), 1, None)[None, :]  # solo estudios con ese slot
    entropy = -np.where(w > 0, w * np.log(np.clip(w, 1e-12, None)), 0.0).sum(-1)  # [N, 12], nats
    n_valid = present.sum(-1, keepdims=True)
    entropy_norm = np.where(n_valid > 1, entropy / np.log(np.maximum(n_valid, 2)), 0.0)
    unit = w / np.clip(np.linalg.norm(w, axis=-1, keepdims=True), 1e-12, None)
    cosine = np.einsum("nis,njs->nij", unit, unit)
    off_diagonal = ~np.eye(len(LABELS), dtype=bool)
    return {
        "n_studies": int(len(w)),
        "slot_coverage": {slot: float(slot_present[:, k].mean()) for k, slot in enumerate(SLOT_NAMES)},
        "slot_mass": {label: {slot: float(slot_mass[i, k]) for k, slot in enumerate(SLOT_NAMES)}
                      for i, label in enumerate(LABELS)},
        "slot_mass_present": {label: {slot: float(slot_mass_present[i, k]) for k, slot in enumerate(SLOT_NAMES)}
                              for i, label in enumerate(LABELS)},
        "plane_mass": {label: {p: float(plane_mass[p][i]) for p in PLANES} for i, label in enumerate(LABELS)},
        "entropy": {label: float(entropy[:, i].mean()) for i, label in enumerate(LABELS)},
        "entropy_norm": {label: float(entropy_norm[:, i].mean()) for i, label in enumerate(LABELS)},
        "mean_pairwise_cosine": float(cosine[:, off_diagonal].mean()),
    }

def attention_long(weights, slots, present, studies, split_name):
    n, l, s = np.nonzero(np.broadcast_to(present[:, None, :], weights.shape))
    slot = slots[n, s]
    return {"study": studies[n], "split": np.full(len(n), split_name), "label": np.array(LABELS)[l],
            "series_index": s, "slot": slot, "slot_name": np.array(SLOT_NAMES)[slot], "plane": SLOT_PLANE[slot],
            "weight": weights[n, l, s]}

ATTENTION = {}
for f in FOLDS:
    _, val_loader, gold_loader = make_loaders(f)
    model = build_model()
    model.load_state_dict(torch.load(RESULTS / f"best_f{f}.pt", map_location=DEVICE)["model"])
    parts, ATTENTION[str(f)] = [], {}
    for split_name, loader in [("val", val_loader), ("gold", gold_loader)]:
        weights, slots, present, studies = collect_attention(model, loader)
        ATTENTION[str(f)][split_name] = attention_summary(weights, slots, present)
        parts.append(attention_long(weights, slots, present, studies, split_name))
    np.savez_compressed(RESULTS / f"attn_f{f}.npz", **{k: np.concatenate([p[k] for p in parts]) for k in parts[0]})
    per_fold[str(f)]["attention"] = ATTENTION[str(f)]  # va a metrics.json (celda 6) en per_fold.<fold>.attention
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    val_summary = ATTENTION[str(f)]["val"]
    print(f"fold {f} · similitud coseno media entre las 12 atenciones (val): {val_summary['mean_pairwise_cosine']:.3f}")
    display(pd.DataFrame(val_summary["slot_mass"]).T.assign(entropy_norm=pd.Series(val_summary["entropy_norm"])).round(3))

## 6. Guardar `results/<EXP>/metrics.json`

In [ ]:
metrics = {
    "schema_version": 1, "exp": EXP, "author": AUTHOR, "parent": PADRE,
    "split": SPLIT, "split_sha256": split.attrs["sha256"], "git": GIT,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "phase": "smoke" if SMOKE else ("confirm" if oof_auc is not None else "screen"),
    "select_on": SELECT_ON, "params": PARAMS, "per_fold": per_fold,
    "gold_auc": gold_auc["mean"], "gold_per_label": gold_auc["per_label"], "oof_pseudo_auc": oof_auc,
    "delta": {"pseudo_f0": delta_pseudo, "gold": delta_gold},
    "decision_auto": DECISION,
    "gpu_hours_total": sum(f["gpu_hours"] for f in per_fold.values()),
}
print("Guardado:", guardar_metricas(RESULTS, metrics))

## 7. Verdict

Copia los números de la celda 5 a la sección 3 de `SPEC.md` y actualiza tu fila en `docs/EXPERIMENTOS.md`
(estado `worked`, `didnt` o `killed` y una línea de qué aprendimos). Después: commit y PR a `main`.